# Task 2 — Profiling and Cleaning

All active sources are cleaned through the shared functions in `src/`. Missing required values are **not** dropped here; they remain for Task 3 so the rejection report is complete. Partial dates are not completed with invented month/day values.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from main import build_kaust, build_kfupm, build_ksu
from src import profile
from src.config import get_paths, load_config, year_range

config = load_config(ROOT)
paths = get_paths(config).ensure()
min_year, max_year = year_range(config)


In [2]:
kaust = build_kaust(config, paths, min_year, max_year)
kfupm = build_kfupm(config, paths, min_year, max_year)
ksu = build_ksu(config, paths, min_year, max_year)

print("KAUST cleaned:", len(kaust))
print("KFUPM cleaned:", len(kfupm))
print("KSU cleaned:", len(ksu))


  KAUST repository cleaned :    928
  KAUST Crossref cleaned   :    124
  KFUPM Pure cleaned       :    441
  KSU 2025 JSON repairs   :      1
  KSU reviewed enrichment  :      4 records
  KSU cleaned              :  41183
KAUST cleaned: 1052
KFUPM cleaned: 441
KSU cleaned: 41183


## Missing-value profile

In [3]:
for name, frame in [("KAUST", kaust), ("KFUPM", kfupm), ("KSU", ksu)]:
    print(f"\n{name}")
    display(profile.missing_report(frame))



KAUST


,column,missing,missing_percent,required
0,research_id,0,0.00,True
1,university,0,0.00,True
2,title,0,0.00,True
3,authors,1,0.10,True
4,publication_year,0,0.00,True
5,publication_date,6,0.57,False
6,abstract,36,3.42,False
7,research_field,1052,100.00,False
8,tech_category,1052,100.00,False
9,journal,331,31.46,False



KFUPM


,column,missing,missing_percent,required
0,research_id,0,0.00,True
1,university,0,0.00,True
2,title,0,0.00,True
3,authors,0,0.00,True
4,publication_year,0,0.00,True
5,publication_date,356,80.73,False
6,abstract,20,4.54,False
7,research_field,299,67.80,False
8,tech_category,441,100.00,False
9,journal,26,5.90,False



KSU


,column,missing,missing_percent,required
0,research_id,0,0.00,True
1,university,0,0.00,True
2,title,0,0.00,True
3,authors,0,0.00,True
4,publication_year,0,0.00,True
5,publication_date,41183,100.00,False
6,abstract,16146,39.21,False
7,research_field,41183,100.00,False
8,tech_category,41183,100.00,False
9,journal,0,0.00,False


## Duplicate and year checks

In [4]:
for name, frame in [("KAUST", kaust), ("KFUPM", kfupm), ("KSU", ksu)]:
    print(f"\n{name}")
    display(profile.duplicate_report(frame, [["research_id"], ["doi"]]))
    display(profile.year_distribution(frame))



KAUST


,columns,duplicate_rows
0,research_id,0
1,doi,0


,publication_year,records
0,2023,928
1,2024,51
2,2025,73



KFUPM


,columns,duplicate_rows
0,research_id,0
1,doi,0


,publication_year,records
0,2023,80
1,2024,87
2,2025,164
3,2026,110



KSU


,columns,duplicate_rows
0,research_id,0
1,doi,8


,publication_year,records
0,2023,12346
1,2024,16134
2,2025,12703


### Cleaning decisions

- Mandatory-field failures are retained for validation rather than silently removed.
- `publication_date` is populated only when the source supplies year, month, and day.
- DOI values are normalized to bare lowercase form.
- KFUPM uses the Pure organisational-unit field for the Computer Engineering department filter.
- KSU `publication_year` comes from the annual source file and its `url` is the dataset URL because no per-publication URL is supplied.
- PNU is excluded from the final dataset; its notebooks are preserved under `notebooks/excluded_pnu/` for audit.